# Public vs Private Collections — preset-driven demo

Two access scenarios driven entirely by the **presets API** — no inline routing or driver config payloads.

- **Scenario A** — apply `public_open_data` preset → ingest via STAC transactions → anonymous and authenticated STAC search both return items.
- **Scenario B** — apply `private_tenant` preset → authenticated user sees items; anonymous search returns 401/403 or 0 features.

Required env vars:
- `DYNASTORE_BASE_URL` — base URL (default `http://localhost:8080`)
- `DYNASTORE_TOKEN` — bearer token for the authenticated client

Optional:
- `DYNASTORE_ANON_BASE_URL` — separate base for anonymous probes (defaults to `DYNASTORE_BASE_URL`)
- `IDP_PUBLIC_URL`, `IDP_CLIENT_ID`, `IDP_CLIENT_SECRET` — auto-provision token via client_credentials

In [ ]:
import json
import os
import time
import uuid

import httpx

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True), override=False)
except Exception:
    pass

# Auto-provision token via IDP client_credentials if not set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ANON_BASE_URL = os.environ.get("DYNASTORE_ANON_BASE_URL", BASE_URL)
TOKEN = os.environ.get("DYNASTORE_TOKEN")
if not TOKEN:
    raise RuntimeError("Set DYNASTORE_TOKEN before running.")

RUN_ID = uuid.uuid4().hex[:6]
PUB_CATALOG_ID = f"demo-public-{RUN_ID}"
PUB_COLL_ID = f"pub-coll-{RUN_ID}"
PRIV_CATALOG_ID = f"demo-private-{RUN_ID}"
PRIV_COLL_ID = f"priv-coll-{RUN_ID}"

JSON_HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
AUTH_HEADERS = {**JSON_HEADERS, "Authorization": f"Bearer {TOKEN}"}

client = httpx.Client(base_url=BASE_URL, headers=AUTH_HEADERS, timeout=30.0)
anon = httpx.Client(base_url=ANON_BASE_URL, headers=JSON_HEADERS, timeout=30.0)

print(f"RUN_ID={RUN_ID}")
print(f"  base: {BASE_URL}")
print(f"  anon: {ANON_BASE_URL}")

## Helpers — readiness and drain polls

In [ ]:
def wait_for_catalog(cat_id, timeout_s=60, poll_s=2.0):
    """Poll GET /stac/catalogs/{cat} until 200/401/403 (tenant provisioned)."""
    deadline = time.monotonic() + timeout_s
    last = (None, "")
    while time.monotonic() < deadline:
        r = client.get(f"/stac/catalogs/{cat_id}")
        last = (r.status_code, r.text[:200])
        if r.status_code in (200, 401, 403):
            return last
        time.sleep(poll_s)
    raise TimeoutError(f"catalog {cat_id} not reachable in {timeout_s}s; last={last}")


def create_collection_when_ready(cat_id, body, timeout_s=120):
    """POST a collection, retrying 409 while the catalog is still provisioning."""
    path = f"/stac/catalogs/{cat_id}/collections"
    deadline = time.monotonic() + timeout_s
    delay, attempt, last = 1.0, 0, None
    while time.monotonic() < deadline:
        attempt += 1
        r = client.post(path, content=json.dumps(body))
        last = r
        if r.is_success:
            print(f"  collection {body['id']!r}: {r.status_code} (attempt {attempt})")
            return r
        if r.status_code == 409:
            print(f"  collection {body['id']!r}: 409 (attempt {attempt}) — retrying in {delay:.1f}s")
            time.sleep(delay)
            delay = min(delay * 1.7, 8.0)
            continue
        raise RuntimeError(f"collection POST failed: {r.status_code} {r.text[:300]}")
    raise TimeoutError(f"collection {body['id']!r}: still 409 after {timeout_s}s")


def wait_for_search(cat_id, expected, timeout_s=90, poll_s=3.0):
    """Poll authenticated STAC search until >= expected features found."""
    deadline = time.monotonic() + timeout_s
    count, last_status, last_body = 0, None, ""
    while time.monotonic() < deadline:
        r = client.post(
            f"/stac/catalogs/{cat_id}/search",
            content=json.dumps({"limit": 50}),
        )
        last_status, last_body = r.status_code, r.text[:300]
        if r.is_success:
            count = len(r.json().get("features", []))
            if count >= expected:
                return count, last_status, last_body
        time.sleep(poll_s)
    return count, last_status, last_body


def diagnose_search_miss(cat_id):
    """Probe PG fallback to determine which layer is missing items."""
    r = client.post(
        f"/stac/catalogs/{cat_id}/search",
        params={"hint": "geometry_exact"},
        content=json.dumps({"limit": 50}),
    )
    pg_count = len(r.json().get("features", [])) if r.is_success else None
    return {"pg_hint_status": r.status_code, "pg_hint_count": pg_count, "body": r.text[:300]}

---
## Scenario A — `public_open_data` preset

The `public_open_data` composite preset wires role seed, IAM policies, public-catalog routing, and STAC/web interfaces. It is a `PLATFORM`-tier preset with `catalog_scopable=True`, so it can be applied at the catalog scope.

### A.1 — Create the catalog

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PUB_CATALOG_ID, "type": "Catalog", "stac_version": "1.0.0",
    "title": f"Demo public {RUN_ID}",
    "description": "Public preset demo — safe to delete",
    "links": [],
}))
assert r.status_code in (200, 201, 409), f"catalog create: {r.status_code} {r.text[:200]}"
print(f"POST /stac/catalogs: {r.status_code}")
wait_for_catalog(PUB_CATALOG_ID)
print(f"Catalog {PUB_CATALOG_ID!r} ready")

### A.2 — Apply `public_open_data` preset at catalog scope

`POST /configs/catalogs/{cat}/presets/public_open_data` — applies the composite bundle at the catalog tier.

In [ ]:
r = client.post(f"/configs/catalogs/{PUB_CATALOG_ID}/presets/public_open_data")
assert r.is_success, f"apply public_open_data failed: {r.status_code} {r.text[:300]}"
print(f"POST /configs/catalogs/{PUB_CATALOG_ID}/presets/public_open_data: {r.status_code}")
print(json.dumps(r.json(), indent=2)[:600])

### A.3 — Create a collection and ingest STAC items

In [ ]:
create_collection_when_ready(PUB_CATALOG_ID, {
    "id": PUB_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Public items demo", "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [[None, None]]},
    },
    "links": [],
})

PUB_ITEM_IDS = []
for i in range(3):
    item_id = f"pub-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PUB_CATALOG_ID}/collections/{PUB_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0", "id": item_id,
            "geometry": {"type": "Point", "coordinates": [12.5 + i * 0.1, 41.9]},
            "bbox": [12.4 + i * 0.1, 41.8, 12.6 + i * 0.1, 42.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z"},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PUB_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code}")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:200]}")

assert len(PUB_ITEM_IDS) == 3, f"only {len(PUB_ITEM_IDS)}/3 items ingested"

### A.4 — Authenticated STAC search returns items

Polls `POST /stac/catalogs/{cat}/search` until the OUTBOX drain completes.

In [ ]:
count, status, _ = wait_for_search(PUB_CATALOG_ID, expected=3, timeout_s=90)
print(f"authenticated STAC search: HTTP {status}, {count} feature(s)")
if count < 3:
    diag = diagnose_search_miss(PUB_CATALOG_ID)
    print(f"diagnostic: {json.dumps(diag, indent=2)}")
    raise RuntimeError(f"expected 3 items, got {count}")
print("  OK — items drained from OUTBOX and visible via STAC search")

### A.5 — Anonymous STAC search also returns items

No `Authorization` header. The `public_open_data` preset opens the catalog to anonymous reads.

In [ ]:
r = anon.post(f"/stac/catalogs/{PUB_CATALOG_ID}/search", content=json.dumps({"limit": 10}))
print(f"anonymous STAC search: HTTP {r.status_code}")
feats = r.json().get("features", []) if r.is_success else []
print(f"  features returned: {len(feats)}")
assert len(feats) >= 3, "public catalog must be searchable without auth"
print("  OK — anonymous search succeeds")

---
## Scenario B — `private_tenant` preset

The `private_tenant` composite preset wires role seed, IAM policies, private-catalog routing, ES-private item routing, and STAC/web interfaces. It is a `PLATFORM`-tier preset with `catalog_scopable=True`.

### B.1 — Create the private catalog

In [ ]:
r = client.post("/stac/catalogs", content=json.dumps({
    "id": PRIV_CATALOG_ID, "type": "Catalog", "stac_version": "1.0.0",
    "title": f"Demo private {RUN_ID}",
    "description": "Private preset demo — safe to delete",
    "links": [],
}))
assert r.status_code in (200, 201, 409), f"create private catalog: {r.status_code} {r.text[:200]}"
print(f"POST /stac/catalogs (private): {r.status_code}")
wait_for_catalog(PRIV_CATALOG_ID)
print(f"Catalog {PRIV_CATALOG_ID!r} ready")

### B.2 — Apply `private_tenant` preset at catalog scope

In [ ]:
r = client.post(f"/configs/catalogs/{PRIV_CATALOG_ID}/presets/private_tenant")
assert r.is_success, f"apply private_tenant failed: {r.status_code} {r.text[:300]}"
print(f"POST /configs/catalogs/{PRIV_CATALOG_ID}/presets/private_tenant: {r.status_code}")
print(json.dumps(r.json(), indent=2)[:600])

### B.3 — Create collection and ingest private items

In [ ]:
create_collection_when_ready(PRIV_CATALOG_ID, {
    "id": PRIV_COLL_ID, "type": "Collection", "stac_version": "1.0.0",
    "description": "Private items demo", "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [[None, None]]},
    },
    "links": [],
})

PRIV_ITEM_IDS = []
for i in range(3):
    item_id = f"priv-item-{i}-{RUN_ID}"
    r = client.post(
        f"/stac/catalogs/{PRIV_CATALOG_ID}/collections/{PRIV_COLL_ID}/items",
        content=json.dumps({
            "type": "Feature", "stac_version": "1.0.0", "id": item_id,
            "geometry": {"type": "Point", "coordinates": [13.5 + i * 0.1, 42.9]},
            "bbox": [13.4 + i * 0.1, 42.8, 13.6 + i * 0.1, 43.0],
            "properties": {"datetime": "2024-06-01T00:00:00Z"},
            "links": [], "assets": {},
        }),
    )
    if r.status_code in (200, 201):
        PRIV_ITEM_IDS.append(item_id)
        print(f"  item {item_id}: {r.status_code}")
    else:
        print(f"  item {item_id}: {r.status_code} {r.text[:200]}")

assert len(PRIV_ITEM_IDS) == 3, f"only {len(PRIV_ITEM_IDS)}/3 private items ingested"

### B.4 — Authenticated user sees all private items

In [ ]:
count, status, _ = wait_for_search(PRIV_CATALOG_ID, expected=3, timeout_s=90)
print(f"authenticated STAC search: HTTP {status}, {count} feature(s)")
if count < 3:
    diag = diagnose_search_miss(PRIV_CATALOG_ID)
    print(f"diagnostic: {json.dumps(diag, indent=2)}")
    raise RuntimeError("authenticated user must see all 3 private items")
print("  OK — authenticated search returns all items")

### B.5 — Anonymous caller is denied

The `private_tenant` preset's IAM bundle gates the catalog's search surface. Anonymous access must return 401/403 or 0 features.

In [ ]:
r = anon.post(f"/stac/catalogs/{PRIV_CATALOG_ID}/search", content=json.dumps({"limit": 10}))
print(f"anonymous STAC search: HTTP {r.status_code}")
if r.status_code in (401, 403):
    print("  OK — anonymous access rejected at the auth layer")
elif r.is_success and not r.json().get("features", []):
    print("  OK — anonymous search returned 0 features (privacy enforced)")
else:
    feats = r.json().get("features", []) if r.is_success else []
    raise RuntimeError(
        f"PRIVACY LEAK: anonymous caller received {len(feats)} feature(s); "
        f"first id: {feats[0].get('id') if feats else None}"
    )

---
## Recap

| Scenario | Preset applied | Authenticated | Anonymous |
|---|---|---|---|
| A — public | `public_open_data` | OK | OK |
| B — private | `private_tenant` | OK | DENY / 0 features |

Both scenarios differ only in which preset is applied — no inline driver or routing config needed.

## Teardown

In [ ]:
for cat_id in [PUB_CATALOG_ID, PRIV_CATALOG_ID]:
    r = client.delete(f"/stac/catalogs/{cat_id}", params={"force": "true"})
    print(f"DELETE {cat_id!r}: {r.status_code}")
client.close()
anon.close()
print("Done.")